In [1]:
import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
from tensorflow.keras.models import load_model
import time
import pyttsx3  # Text-to-Speech

# Load the trained model
model = load_model("sign_language_model_mobilenet.h5")

# Define class labels (should match your training dataset order)
CATEGORIES = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M",
              "N", "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z",
              "space", "delete", "nothing"]

# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7)

# Initialize Text-to-Speech
engine = pyttsx3.init()

# Open webcam
cap = cv2.VideoCapture(0)

sentence = ""  # Store predicted letters
last_pred = None  # Last predicted letter
last_time = time.time()  # Timer for letter confirmation
confidence_display = None  # Store confidence percentage

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Flip the frame horizontally (for natural interaction)
    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    # Convert frame to RGB (MediaPipe requires RGB)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Detect hands
    results = hands.process(rgb_frame)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # Get bounding box around hand
            x_min, y_min, x_max, y_max = w, h, 0, 0

            for landmark in hand_landmarks.landmark:
                x, y = int(landmark.x * w), int(landmark.y * h)
                x_min = min(x_min, x)
                y_min = min(y_min, y)
                x_max = max(x_max, x)
                y_max = max(y_max, y)

            # Expand bounding box slightly
            padding = 20
            x_min = max(x_min - padding, 0)
            y_min = max(y_min - padding, 0)
            x_max = min(x_max + padding, w)
            y_max = min(y_max + padding, h)

            # Extract hand region
            hand_img = frame[y_min:y_max, x_min:x_max]

            if hand_img.shape[0] > 0 and hand_img.shape[1] > 0:
                # Resize and normalize image for the model
                hand_img = cv2.resize(hand_img, (128, 128))
                hand_img = hand_img / 255.0
                hand_img = np.expand_dims(hand_img, axis=0)

                # Predict sign language gesture
                predictions = model.predict(hand_img)
                predicted_index = np.argmax(predictions)
                predicted_label = CATEGORIES[predicted_index]
                confidence = predictions[0][predicted_index]

                # Show confidence score temporarily
                confidence_display = f"{predicted_label}: {confidence*100:.2f}%"

                # Adjust confidence threshold for better space/delete detection
                min_confidence = 0.8  # Default confidence threshold
                if predicted_label in ["space", "delete"]:
                    min_confidence = 0.6  # Allow lower confidence for space & delete

                # Check if prediction confidence is above threshold
                if confidence > min_confidence:
                    current_time = time.time()

                    if predicted_label != last_pred:
                        last_pred = predicted_label
                        last_time = current_time  # Reset timer

                    # Confirm prediction only if stable for 2 seconds
                    if current_time - last_time >= 2:
                        if predicted_label == "space":
                            sentence += " "  # Add space
                        elif predicted_label == "delete":
                            sentence = sentence[:-1]  # Remove last character
                        elif predicted_label != "nothing":
                            sentence += predicted_label  # Add letter
                        
                        last_pred = None  # Reset for next input
                        last_time = current_time
                        confidence_display = None  # Remove confidence display

                # Draw bounding box
                cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

            # Draw hand landmarks
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

    # Display confidence score temporarily
    if confidence_display:
        cv2.putText(frame, confidence_display, (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Display recognized sentence
    cv2.putText(frame, "Sentence: " + sentence, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    cv2.imshow("Sign Language Recognition", frame)

    # Exit on pressing 'q' and speak sentence
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Final Sentence:", sentence)
        engine.say(sentence)  # Speak sentence
        engine.runAndWait()
        break

# Release resources
cap.release()
cv2.destroyAllWindows()

1/1 [==============================] - 0s 17ms/step
Final Sentence: VVZ
